# KapInstruct-100M: Dataset Builder & Hugging Face / Kaggle Publisher

This notebook provides an automated, reproducible workflow to build and publish **KapInstruct-100M** (100,000,000 final usable content tokens) directly to the Hugging Face Hub and Kaggle Datasets.

### Pipeline Overview
- **Target Token Budget**: 100,000,000 post-filtering usable content tokens
- **Tokenizer**: `Qwen/Qwen3.5-0.8B-Base` (ChatML format)
- **Loss Policy**: `assistant_only` (prompt tokens and template syntax are masked to `-100`; loss is computed exclusively on assistant turns)
- **Mixture**: 12 curated, balanced instruction sources across general instruction, programming, math CoT, STEM QA, and debugging
- **Format**: Chunked PyArrow IPC Shards (`shard_00000.arrow` ... `shard_00024.arrow`) packed into 4096-token sequences with full `manifest.json`, `licenses.json`, and SHA-256 checksums.

## Step 1: Install Dependencies & Authenticate Hugging Face

In [ ]:
!pip install -q --upgrade datasets transformers pyarrow pyyaml huggingface_hub

import os
import sys
import json
import glob
import pyarrow as pa
from pathlib import Path
from huggingface_hub import HfApi, login

# Authenticate with Hugging Face using Kaggle Secrets or token
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    print(" Successfully logged into Hugging Face Hub via Kaggle Secrets.")
except Exception as e:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    print(" Successfully logged into Hugging Face Hub.")

## Step 2: Clone or Setup Repository

In [ ]:
repo_path = Path.cwd()
if not (repo_path / "configs" / "kapinstruct_dataset_config.yaml").exists():
    !git clone https://github.com/rudy-07/QaptaanLM-0.75B.git repo
    os.chdir("repo")
    
print(f"Working directory: {Path.cwd()}")
assert Path("configs/kapinstruct_dataset_config.yaml").exists(), "Configuration file not found!"

## Step 3: Run Unit Tests & End-to-End Smoke Test

In [ ]:
print("=== Running Unit Tests ===")
!python -u tests/test_kapinstruct.py

print("\n=== Running End-to-End Smoke Test Across All 12 Sources ===")
!python -u scripts/04_kapinstruct_smoke_test.py

## Step 4: Build KapInstruct-100M Dataset (Streaming Build)

This runs the streaming build pipeline, normalizing 12 datasets, filtering, deduplicating, tokenizing with assistant-only loss masking, deficit-scheduling, and packing into Arrow shards.

In [ ]:
OUTPUT_DIR = "/kaggle/working/data/kapinstruct"
TARGET_TOKENS = 100_000_000  # 100M usable content tokens

# If dataset is already built and available on Hugging Face, you can fast-download it in 15 seconds:
# from huggingface_hub import snapshot_download
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

!python -u scripts/03_build_kapinstruct.py \
    --config configs/kapinstruct_dataset_config.yaml \
    --target-tokens {TARGET_TOKENS} \
    --output-dir {OUTPUT_DIR} \
    --loss-policy assistant_only

## Step 5: Verify Generated Shards, Manifest, and Loss Alignment

In [ ]:
from transformers import AutoTokenizer

manifest_path = Path(OUTPUT_DIR) / "manifest.json"
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print("=== Manifest Summary ===")
print(f"Dataset:          {manifest['dataset_name']}")
print(f"Rendered Tokens:  {manifest['achieved_rendered_tokens']:,}")
print(f"Trainable Tokens: {manifest['achieved_trainable_tokens']:,}")
print(f"Total Documents:  {manifest['total_documents']:,}")
print(f"Total Shards:     {manifest['num_shards']}")
print(f"Total Sequences:  {manifest['total_sequences']:,}")

# Inspect first shard
shards = sorted(glob.glob(f"{OUTPUT_DIR}/shard_*.arrow"))
print(f"\nFound {len(shards)} Arrow shards.")

with open(shards[0], "rb") as f:
    reader = pa.ipc.open_file(f)
    table = reader.read_all()

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B-Base")
inp_ids = table["input_ids"][0].as_py()
labels = table["labels"][0].as_py()

print(f"\nFirst sequence length: {len(inp_ids)}")
print(f"Trainable tokens in seq 0: {sum(1 for l in labels if l != -100)}")
print(f"Masked tokens in seq 0:    {sum(1 for l in labels if l == -100)}")
print("\nDecoded Sequence Preview (first 200 tokens):")
print(tok.decode(inp_ids[:200]))

## Step 6: Publish Dataset to Hugging Face Hub

In [ ]:
HF_DATASET_REPO = "kaptaan45/KapInstruct-100M"

!python -u scripts/publish_hf_kapinstruct.py \
    --data-dir {OUTPUT_DIR} \
    --hf-repo {HF_DATASET_REPO}

## Step 7: Push Dataset Version to Kaggle Dataset (kaptaan45/kapinstruct-100m)

In [ ]:
KAGGLE_DATASET = "kaptaan45/kapinstruct-100m"

# Create dataset-metadata.json for Kaggle
meta = {
    "title": "KapInstruct-100M",
    "id": KAGGLE_DATASET,
    "licenses": [{"name": "other"}]
}
with open(f"{OUTPUT_DIR}/dataset-metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

# Push new dataset version to Kaggle
!kaggle datasets version -p {OUTPUT_DIR} -m "Upload complete 25 Arrow shards, manifests, and reports" --dir-mode zip

# Update Kaggle description & schemas
!python -u scripts/publish_kaggle_dataset_kapinstruct.py